# Phase 6 — Controlled Retrieval Ablation

This experiment isolates the contribution of:

1. Semantic retrieval
2. BM25 / Hybrid retrieval
3. Cross-encoder reranking

All experiments use:

- The same 96 clean chunks
- The same embedding model: BAAI/bge-small-en-v1.5
- The same 116 short queries
- The same 41 long queries
- The same Top-K settings

Configurations:

- A. Semantic only
- B. Hybrid only
- C. Semantic + Reranker
- D. Hybrid + Reranker

## Import

In [3]:
import json
import re
import numpy as np
import pandas as pd

from pathlib import Path
from collections import defaultdict

from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

print("Libraries loaded.")

Libraries loaded.


## Cell 1 — Load the existing clean chunks

Use your existing chunk file/path from the previous notebook. If your variable is already available because you're continuing in the same notebook, skip this cell.

In [7]:
from pathlib import Path
import json
import importlib.util

BASE_DIR = Path(".")

# --------------------------------------------------
# Clean chunk file
# --------------------------------------------------

CHUNKS_FILE = (
    BASE_DIR
    / "chunked_data"
    / "chunks.jsonl"
)

# --------------------------------------------------
# Evaluation queries
# --------------------------------------------------

SHORT_QUERIES_FILE = (
    BASE_DIR
    / "queries"
    / "short_queries_116.py"
)

LONG_QUERIES_FILE = (
    BASE_DIR
    / "queries"
    / "long_queries_41.py"
)

# --------------------------------------------------
# Output directory
# --------------------------------------------------

OUTPUT_DIR = BASE_DIR / "phase6_ablation_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Paths configured.")
print("Chunks:", CHUNKS_FILE)
print("Short queries:", SHORT_QUERIES_FILE)
print("Long queries:", LONG_QUERIES_FILE)
print("Output:", OUTPUT_DIR)


# ==================================================
# Helper function to load query lists from .py files
# ==================================================

def load_python_variable(file_path, variable_name):
    """
    Load a Python file and extract a specific variable from it.
    """

    spec = importlib.util.spec_from_file_location(
        "query_module",
        file_path
    )

    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)

    if not hasattr(module, variable_name):
        raise AttributeError(
            f"Variable '{variable_name}' not found in {file_path}"
        )

    return getattr(module, variable_name)


# ==================================================
# Load short queries
# ==================================================

short_queries = load_python_variable(
    SHORT_QUERIES_FILE,
    "evaluation_queries"
)

print(f"\nLoaded {len(short_queries)} short queries.")


# ==================================================
# Load long queries
# ==================================================

long_queries = load_python_variable(
    LONG_QUERIES_FILE,
    "evaluation_queries_long"
)

print(f"Loaded {len(long_queries)} long queries.")


# ==================================================
# Verify data
# ==================================================

print("\nFirst short query:")
print("Query:", short_queries[0]["query"])
print("Expected document:", short_queries[0]["expected_document"])

print("\nFirst long query:")
print("Query:", long_queries[0]["query"])
print("Expected document:", long_queries[0]["expected_document"])

Paths configured.
Chunks: chunked_data\chunks.jsonl
Short queries: queries\short_queries_116.py
Long queries: queries\long_queries_41.py
Output: phase6_ablation_outputs

Loaded 116 short queries.
Loaded 41 long queries.

First short query:
Query: How much vacation time do employees get?
Expected document: Vacation and Sick Leave.md

First long query:
Query: I've been working at Clef for about six months now and I'm wondering how many vacation days I've accumulated so far and whether unused sick days carry over to the next year
Expected document: Vacation and Sick Leave.md


## Cell 4 — Verify files

In [8]:
required_files = [
    CHUNKS_FILE,
    SHORT_QUERIES_FILE,
    LONG_QUERIES_FILE,
]

for path in required_files:
    print(f"{path}: {'FOUND' if path.exists() else 'NOT FOUND'}")

assert CHUNKS_FILE.exists(), f"Missing: {CHUNKS_FILE}"
assert SHORT_QUERIES_FILE.exists(), f"Missing: {SHORT_QUERIES_FILE}"
assert LONG_QUERIES_FILE.exists(), f"Missing: {LONG_QUERIES_FILE}"

print("\nAll required files found.")

chunked_data\chunks.jsonl: FOUND
queries\short_queries_116.py: FOUND
queries\long_queries_41.py: FOUND

All required files found.


## Cell 5 — Load JSON helper

In [12]:
chunks = []

with open(CHUNKS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            chunks.append(json.loads(line))

print("Total chunks loaded:", len(chunks))

Total chunks loaded: 96


In [13]:

# print("Chunks:", len(chunks))
print("Short queries:", len(short_queries))
print("Long queries:", len(long_queries))
print("Total queries:", len(short_queries) + len(long_queries))

Short queries: 116
Long queries: 41
Total queries: 157


## Cell 6 — Validate the clean corpus

This makes sure README is actually absent.

In [14]:
readme_chunks = [
    c for c in chunks
    if "README.md" in str(c.get("document", ""))
    or "README.md" in str(c.get("file_path", ""))
]

print("README chunks:", len(readme_chunks))

assert len(chunks) == 96
assert len(readme_chunks) == 0

print("✓ Clean corpus contains 96 chunks.")
print("✓ README.md is completely absent.")

README chunks: 0
✓ Clean corpus contains 96 chunks.
✓ README.md is completely absent.


## Cell 7 — Validate queries

In [16]:
assert len(short_queries) == 116
assert len(long_queries) == 41

print("✓ Short evaluation set: 116 queries")
print("✓ Long evaluation set: 41 queries")
print("✓ Total evaluation queries: 157")

✓ Short evaluation set: 116 queries
✓ Long evaluation set: 41 queries
✓ Total evaluation queries: 157


## Cell 2 — Load the same embedding model

Again, if your existing notebook already has model, embeddings, etc., don't duplicate it.

In [17]:

# Use exactly the same embedding model as the previous experiments.
EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print("Embedding model loaded:")
print(EMBEDDING_MODEL_NAME)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1424.52it/s]


Embedding model loaded:
BAAI/bge-small-en-v1.5


## Cell 9 — Prepare chunk texts

In [18]:
chunk_texts = [
    chunk["content"]
    for chunk in chunks
]

chunk_ids = [
    chunk["chunk_id"]
    for chunk in chunks
]

print("Number of chunk texts:", len(chunk_texts))
print("Unique chunk IDs:", len(set(chunk_ids)))

assert len(chunk_texts) == 96
assert len(set(chunk_ids)) == 96

Number of chunk texts: 96
Unique chunk IDs: 96


## Cell 10 — Generate chunk embeddings

In [19]:
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

chunk_embeddings = np.asarray(chunk_embeddings)

print("Embedding shape:", chunk_embeddings.shape)

assert chunk_embeddings.shape[0] == 96

Batches: 100%|██████████| 3/3 [00:14<00:00,  4.99s/it]

Embedding shape: (96, 384)


## Cell 11 — Build chunk lookup

In [21]:
chunk_lookup = {
    chunk["chunk_id"]: chunk
    for chunk in chunks
}

assert len(chunk_lookup) == 96

print("Chunk lookup ready.")

Chunk lookup ready.


## Cell 12 — Semantic retrieval

We will retrieve a larger candidate pool first.

The final Top-5 will be selected after ranking.

In [22]:
SEMANTIC_CANDIDATES = 20
FINAL_TOP_K = 5

def semantic_retrieve(query, top_k=SEMANTIC_CANDIDATES):
    """
    Dense semantic retrieval using BGE embeddings.
    """

    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    scores = np.dot(
        chunk_embeddings,
        query_embedding
    )

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for rank, idx in enumerate(top_indices, start=1):

        chunk = chunks[idx]

        results.append({
            "chunk_id": chunk["chunk_id"],
            "document": chunk["document"],
            "file_path": chunk.get("file_path"),
            "category": chunk.get("category"),
            "section_path": chunk.get("section_path"),
            "section_title": chunk.get("section_title"),
            "content": chunk["content"],
            "semantic_score": float(scores[idx]),
            "candidate_rank": rank
        })

    return results

## Cell 13 — Build BM25 index

In [23]:
def tokenize(text):
    return re.findall(
        r"\b\w+\b",
        text.lower()
    )


tokenized_chunks = [
    tokenize(text)
    for text in chunk_texts
]

bm25 = BM25Okapi(tokenized_chunks)

print("BM25 index built.")
print("Indexed chunks:", len(tokenized_chunks))

BM25 index built.
Indexed chunks: 96


## Cell 14 — BM25 retrieval

In [24]:
BM25_CANDIDATES = 20

def bm25_retrieve(query, top_k=BM25_CANDIDATES):

    query_tokens = tokenize(query)

    scores = bm25.get_scores(query_tokens)

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for rank, idx in enumerate(top_indices, start=1):

        chunk = chunks[idx]

        results.append({
            "chunk_id": chunk["chunk_id"],
            "document": chunk["document"],
            "file_path": chunk.get("file_path"),
            "category": chunk.get("category"),
            "section_path": chunk.get("section_path"),
            "section_title": chunk.get("section_title"),
            "content": chunk["content"],
            "bm25_score": float(scores[idx]),
            "candidate_rank": rank
        })

    return results

## Cell 15 — Hybrid retrieval

For the ablation, we use the same general semantic + BM25 combination used in the existing improved pipeline.

We will explicitly keep the weight visible so it can be inspected.

In [25]:
SEMANTIC_WEIGHT = 0.5
BM25_WEIGHT = 0.5

def min_max_normalize(scores):

    scores = np.asarray(scores, dtype=float)

    if len(scores) == 0:
        return scores

    min_score = scores.min()
    max_score = scores.max()

    if max_score == min_score:
        return np.zeros_like(scores)

    return (
        (scores - min_score)
        / (max_score - min_score)
    )


def hybrid_retrieve(
    query,
    semantic_top_k=SEMANTIC_CANDIDATES,
    bm25_top_k=BM25_CANDIDATES,
    final_top_k=5
):

    semantic_results = semantic_retrieve(
        query,
        top_k=semantic_top_k
    )

    bm25_results = bm25_retrieve(
        query,
        top_k=bm25_top_k
    )

    candidates = {}

    # ---------------------------------------------
    # Semantic candidates
    # ---------------------------------------------

    for item in semantic_results:

        chunk_id = item["chunk_id"]

        candidates[chunk_id] = {
            **item,
            "bm25_score": 0.0
        }

    # ---------------------------------------------
    # BM25 candidates
    # ---------------------------------------------

    for item in bm25_results:

        chunk_id = item["chunk_id"]

        if chunk_id not in candidates:

            candidates[chunk_id] = {
                **item,
                "semantic_score": 0.0
            }

        else:

            candidates[chunk_id]["bm25_score"] = (
                item["bm25_score"]
            )

    candidates_list = list(
        candidates.values()
    )

    semantic_scores = [
        item["semantic_score"]
        for item in candidates_list
    ]

    bm25_scores = [
        item["bm25_score"]
        for item in candidates_list
    ]

    normalized_semantic = min_max_normalize(
        semantic_scores
    )

    normalized_bm25 = min_max_normalize(
        bm25_scores
    )

    for i, item in enumerate(candidates_list):

        item["normalized_semantic_score"] = float(
            normalized_semantic[i]
        )

        item["normalized_bm25_score"] = float(
            normalized_bm25[i]
        )

        item["hybrid_score"] = (
            SEMANTIC_WEIGHT
            * normalized_semantic[i]
            +
            BM25_WEIGHT
            * normalized_bm25[i]
        )

    candidates_list.sort(
        key=lambda x: x["hybrid_score"],
        reverse=True
    )

    final_results = []

    for rank, item in enumerate(
        candidates_list[:final_top_k],
        start=1
    ):

        item = dict(item)
        item["rank"] = rank

        final_results.append(item)

    return final_results

## Cell 16 — Load cross-encoder reranker

We need to use the same reranker model from your existing improved pipeline.

Your previous retrieval outputs show that the reranker produces rerank_score; the model itself was already part of the Hybrid + Reranker pipeline.

Use the exact model name from your Phase 4 notebook here:

In [26]:
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

reranker = CrossEncoder(
    RERANKER_MODEL_NAME
)

print("Reranker loaded:")
print(RERANKER_MODEL_NAME)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2179.68it/s]


Reranker loaded:
cross-encoder/ms-marco-MiniLM-L-6-v2


## Cell 17 — Semantic + Reranker

In [27]:
def semantic_reranker_retrieve(
    query,
    semantic_top_k=SEMANTIC_CANDIDATES,
    final_top_k=5
):

    candidates = semantic_retrieve(
        query,
        top_k=semantic_top_k
    )

    if not candidates:
        return []

    pairs = [
        [
            query,
            item["content"]
        ]
        for item in candidates
    ]

    scores = reranker.predict(
        pairs
    )

    for item, score in zip(
        candidates,
        scores
    ):
        item["rerank_score"] = float(score)

    candidates.sort(
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    final_results = []

    for rank, item in enumerate(
        candidates[:final_top_k],
        start=1
    ):

        item = dict(item)
        item["rank"] = rank

        final_results.append(item)

    return final_results

## Cell 18 — Hybrid + Reranker

In [28]:
def hybrid_reranker_retrieve(
    query,
    semantic_top_k=SEMANTIC_CANDIDATES,
    bm25_top_k=BM25_CANDIDATES,
    final_top_k=5
):

    candidates = hybrid_retrieve(
        query,
        semantic_top_k=semantic_top_k,
        bm25_top_k=bm25_top_k,
        final_top_k=max(
            semantic_top_k,
            bm25_top_k
        )
    )

    if not candidates:
        return []

    pairs = [
        [
            query,
            item["content"]
        ]
        for item in candidates
    ]

    scores = reranker.predict(
        pairs
    )

    for item, score in zip(
        candidates,
        scores
    ):
        item["rerank_score"] = float(score)

    candidates.sort(
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    final_results = []

    for rank, item in enumerate(
        candidates[:final_top_k],
        start=1
    ):

        item = dict(item)
        item["rank"] = rank

        final_results.append(item)

    return final_results

## Cell 19 — Test one query

Before running all 157 queries, test every configuration on one query.

In [29]:
test_query = short_queries[0]["query"]

print("TEST QUERY:")
print(test_query)

print("\nA — Semantic")
semantic_test = semantic_retrieve(test_query, top_k=5)

for item in semantic_test:
    print(
        item["rank"] if "rank" in item else item["candidate_rank"],
        item["document"]
    )

print("\nB — Hybrid")
hybrid_test = hybrid_retrieve(test_query, final_top_k=5)

for item in hybrid_test:
    print(
        item["rank"],
        item["document"]
    )

print("\nC — Semantic + Reranker")
semantic_rerank_test = semantic_reranker_retrieve(
    test_query,
    final_top_k=5
)

for item in semantic_rerank_test:
    print(
        item["rank"],
        item["document"]
    )

print("\nD — Hybrid + Reranker")
hybrid_rerank_test = hybrid_reranker_retrieve(
    test_query,
    final_top_k=5
)

for item in hybrid_rerank_test:
    print(
        item["rank"],
        item["document"]
    )

TEST QUERY:
How much vacation time do employees get?

A — Semantic
1 Vacation and Sick Leave.md
2 Sabbatical.md
3 New Parent Leave.md
4 Other Protected Absences.md
5 Working Remotely.md

B — Hybrid
1 Vacation and Sick Leave.md
2 New Parent Leave.md
3 Sabbatical.md
4 Welcome to Clef.md
5 Referral Bonuses.md

C — Semantic + Reranker
1 Vacation and Sick Leave.md
2 New Parent Leave.md
3 Sabbatical.md
4 Other Protected Absences.md
5 Other Protected Absences.md

D — Hybrid + Reranker
1 Vacation and Sick Leave.md
2 New Parent Leave.md
3 Sabbatical.md
4 Other Protected Absences.md
5 Other Protected Absences.md


## Cell 20 — Evaluation helper

In [36]:
def get_expected_document(query_item):
    """
    Supports the query format used by the evaluation files.
    """

    if "expected_document" in query_item:
        return query_item["expected_document"]

    if "expected" in query_item:
        return query_item["expected"]

    raise KeyError(
        "Could not find expected document field."
    )


def evaluate_results(results):

    total = len(results)

    top1 = 0
    top3 = 0
    top5 = 0

    details = []

    for item in results:

        expected = item["expected_document"]

        retrieved_documents = [
            result["document"]
            for result in item["results"]
        ]

        rank = 999

        for i, document in enumerate(
            retrieved_documents,
            start=1
        ):
            if document == expected:
                rank = i
                break

        if rank <= 1:
            top1 += 1

        if rank <= 3:
            top3 += 1

        if rank <= 5:
            top5 += 1

        details.append({
            "query_id": item["query_id"],
            "query": item["query"],
            "expected_document": expected,
            "rank": rank,
            "top1": rank <= 1,
            "top3": rank <= 3,
            "top5": rank <= 5
        })

    metrics = {
        "total_queries": total,
        "top1_accuracy": top1 / total,
        "top3_recall": top3 / total,
        "top5_recall": top5 / total,
        "top1_correct": top1,
        "top3_correct": top3,
        "top5_correct": top5
    }

    return metrics, details

## Cell 21 — Run one configuration over a query set

In [31]:
def run_configuration(
    queries,
    method_name
):

    output = []

    for i, query_item in enumerate(
        queries,
        start=1
    ):

        query = query_item["query"]

        expected = get_expected_document(
            query_item
        )

        if method_name == "semantic":

            results = semantic_retrieve(
                query,
                top_k=5
            )

        elif method_name == "hybrid":

            results = hybrid_retrieve(
                query,
                final_top_k=5
            )

        elif method_name == "semantic_reranker":

            results = semantic_reranker_retrieve(
                query,
                final_top_k=5
            )

        elif method_name == "hybrid_reranker":

            results = hybrid_reranker_retrieve(
                query,
                final_top_k=5
            )

        else:
            raise ValueError(
                f"Unknown method: {method_name}"
            )

        output.append({
            "query_id": query_item.get(
                "query_id",
                i
            ),
            "query": query,
            "query_type": query_item.get(
                "query_type"
            ),
            "expected_document": expected,
            "results": results
        })

        print(
            f"\r{method_name}: "
            f"{i}/{len(queries)}",
            end=""
        )

    print()

    return output

## Cell 22 — Define the four experiments

In [32]:
METHODS = {
    "A_semantic": "semantic",
    "B_hybrid": "hybrid",
    "C_semantic_reranker": "semantic_reranker",
    "D_hybrid_reranker": "hybrid_reranker"
}

print("Experiments:")
for label, method in METHODS.items():
    print(f"{label}: {method}")

Experiments:
A_semantic: semantic
B_hybrid: hybrid
C_semantic_reranker: semantic_reranker
D_hybrid_reranker: hybrid_reranker


## Cell 23 — Run all four on short queries
This will take some time because the reranker is evaluated multiple times.

In [33]:
short_results = {}

for label, method in METHODS.items():

    print("\n" + "=" * 70)
    print(label)
    print("=" * 70)

    short_results[label] = run_configuration(
        short_queries,
        method
    )


A_semantic
semantic: 116/116

B_hybrid
hybrid: 116/116

C_semantic_reranker
semantic_reranker: 116/116

D_hybrid_reranker
hybrid_reranker: 116/116


## Cell 24 — Run all four on long queries

In [34]:
long_results = {}

for label, method in METHODS.items():

    print("\n" + "=" * 70)
    print(label)
    print("=" * 70)

    long_results[label] = run_configuration(
        long_queries,
        method
    )


A_semantic
semantic: 41/41

B_hybrid
hybrid: 41/41

C_semantic_reranker
semantic_reranker: 41/41

D_hybrid_reranker
hybrid_reranker: 41/41


## Cell 25 — Evaluate all short-query experiments

In [37]:
short_metrics = {}

for label, results in short_results.items():

    metrics, details = evaluate_results(
        results
    )

    short_metrics[label] = metrics

print(
    pd.DataFrame(short_metrics).T
)

                     total_queries  top1_accuracy  top3_recall  top5_recall  \
A_semantic                   116.0       0.784483     0.879310     0.896552   
B_hybrid                     116.0       0.715517     0.913793     0.931034   
C_semantic_reranker          116.0       0.836207     0.887931     0.896552   
D_hybrid_reranker            116.0       0.827586     0.887931     0.896552   

                     top1_correct  top3_correct  top5_correct  
A_semantic                   91.0         102.0         104.0  
B_hybrid                     83.0         106.0         108.0  
C_semantic_reranker          97.0         103.0         104.0  
D_hybrid_reranker            96.0         103.0         104.0  


## Cell 26 — Evaluate all long-query experiments

In [38]:
long_metrics = {}

for label, results in long_results.items():

    metrics, details = evaluate_results(
        results
    )

    long_metrics[label] = metrics

print(
    pd.DataFrame(long_metrics).T
)

                     total_queries  top1_accuracy  top3_recall  top5_recall  \
A_semantic                    41.0       0.878049      0.95122      0.95122   
B_hybrid                      41.0       0.878049      1.00000      1.00000   
C_semantic_reranker           41.0       0.902439      0.97561      0.97561   
D_hybrid_reranker             41.0       0.902439      0.97561      0.97561   

                     top1_correct  top3_correct  top5_correct  
A_semantic                   36.0          39.0          39.0  
B_hybrid                     36.0          41.0          41.0  
C_semantic_reranker          37.0          40.0          40.0  
D_hybrid_reranker            37.0          40.0          40.0  


## Cell 27 — Clean comparison table

In [39]:
comparison_rows = []

for label, metrics in short_metrics.items():

    comparison_rows.append({
        "Dataset": "Short (116)",
        "Method": label,
        "Top-1 (%)": metrics["top1_accuracy"] * 100,
        "Top-3 (%)": metrics["top3_recall"] * 100,
        "Top-5 (%)": metrics["top5_recall"] * 100
    })

for label, metrics in long_metrics.items():

    comparison_rows.append({
        "Dataset": "Long (41)",
        "Method": label,
        "Top-1 (%)": metrics["top1_accuracy"] * 100,
        "Top-3 (%)": metrics["top3_recall"] * 100,
        "Top-5 (%)": metrics["top5_recall"] * 100
    })


comparison_df = pd.DataFrame(
    comparison_rows
)

comparison_df.round(2)

,Dataset,Method,Top-1 (%),Top-3 (%),Top-5 (%)
0,Short (116),A_semantic,78.45,87.93,89.66
1,Short (116),B_hybrid,71.55,91.38,93.10
2,Short (116),C_semantic_reranker,83.62,88.79,89.66
3,Short (116),D_hybrid_reranker,82.76,88.79,89.66
4,Long (41),A_semantic,87.80,95.12,95.12
5,Long (41),B_hybrid,87.80,100.00,100.00
6,Long (41),C_semantic_reranker,90.24,97.56,97.56
7,Long (41),D_hybrid_reranker,90.24,97.56,97.56


## Cell 28 — Show the contribution of each component

This is the most important analysis table.

In [40]:
def component_analysis(metrics):

    semantic_top1 = (
        metrics["A_semantic"]["top1_accuracy"]
        * 100
    )

    hybrid_top1 = (
        metrics["B_hybrid"]["top1_accuracy"]
        * 100
    )

    semantic_rerank_top1 = (
        metrics["C_semantic_reranker"]["top1_accuracy"]
        * 100
    )

    hybrid_rerank_top1 = (
        metrics["D_hybrid_reranker"]["top1_accuracy"]
        * 100
    )

    return pd.DataFrame([
        {
            "Comparison": "Hybrid contribution",
            "From": "Semantic",
            "To": "Hybrid",
            "Δ Top-1": hybrid_top1 - semantic_top1
        },
        {
            "Comparison": "Reranker contribution on semantic",
            "From": "Semantic",
            "To": "Semantic + Reranker",
            "Δ Top-1": semantic_rerank_top1 - semantic_top1
        },
        {
            "Comparison": "Reranker contribution on hybrid",
            "From": "Hybrid",
            "To": "Hybrid + Reranker",
            "Δ Top-1": hybrid_rerank_top1 - hybrid_top1
        },
        {
            "Comparison": "Total improvement",
            "From": "Semantic",
            "To": "Hybrid + Reranker",
            "Δ Top-1": hybrid_rerank_top1 - semantic_top1
        }
    ])


print("SHORT QUERY COMPONENT ANALYSIS")
component_analysis(short_metrics)

SHORT QUERY COMPONENT ANALYSIS


,Comparison,From,To,Δ Top-1
0,Hybrid contribution,Semantic,Hybrid,-6.896552
1,Reranker contribution on semantic,Semantic,Semantic + Reranker,5.172414
2,Reranker contribution on hybrid,Hybrid,Hybrid + Reranker,11.206897
3,Total improvement,Semantic,Hybrid + Reranker,4.310345


## Cell 29 — Long-query component analysis

In [41]:
print("LONG QUERY COMPONENT ANALYSIS")
component_analysis(long_metrics)

LONG QUERY COMPONENT ANALYSIS


,Comparison,From,To,Δ Top-1
0,Hybrid contribution,Semantic,Hybrid,0.000000
1,Reranker contribution on semantic,Semantic,Semantic + Reranker,2.439024
2,Reranker contribution on hybrid,Hybrid,Hybrid + Reranker,2.439024
3,Total improvement,Semantic,Hybrid + Reranker,2.439024


## Cell 30 — Compare Top-3 and Top-5 contributions

In [42]:
def full_component_analysis(metrics):

    rows = []

    metric_names = {
        "Top-1": "top1_accuracy",
        "Top-3": "top3_recall",
        "Top-5": "top5_recall"
    }

    for display_name, key in metric_names.items():

        semantic = (
            metrics["A_semantic"][key] * 100
        )

        hybrid = (
            metrics["B_hybrid"][key] * 100
        )

        semantic_rerank = (
            metrics["C_semantic_reranker"][key]
            * 100
        )

        hybrid_rerank = (
            metrics["D_hybrid_reranker"][key]
            * 100
        )

        rows.append({
            "Metric": display_name,
            "Semantic": semantic,
            "Hybrid": hybrid,
            "Semantic + Reranker": semantic_rerank,
            "Hybrid + Reranker": hybrid_rerank,
            "Hybrid Δ": hybrid - semantic,
            "Reranker Δ on Semantic":
                semantic_rerank - semantic,
            "Reranker Δ on Hybrid":
                hybrid_rerank - hybrid,
            "Total Δ":
                hybrid_rerank - semantic
        })

    return pd.DataFrame(rows)


print("SHORT")
display(
    full_component_analysis(short_metrics).round(2)
)

print("\nLONG")
display(
    full_component_analysis(long_metrics).round(2)
)

SHORT


,Metric,Semantic,Hybrid,Semantic + Reranker,Hybrid + Reranker,Hybrid Δ,Reranker Δ on Semantic,Reranker Δ on Hybrid,Total Δ
0,Top-1,78.45,71.55,83.62,82.76,-6.90,5.17,11.21,4.31
1,Top-3,87.93,91.38,88.79,88.79,3.45,0.86,-2.59,0.86
2,Top-5,89.66,93.10,89.66,89.66,3.45,0.00,-3.45,0.00



LONG


,Metric,Semantic,Hybrid,Semantic + Reranker,Hybrid + Reranker,Hybrid Δ,Reranker Δ on Semantic,Reranker Δ on Hybrid,Total Δ
0,Top-1,87.80,87.8,90.24,90.24,0.00,2.44,2.44,2.44
1,Top-3,95.12,100.0,97.56,97.56,4.88,2.44,-2.44,2.44
2,Top-5,95.12,100.0,97.56,97.56,4.88,2.44,-2.44,2.44


## Cell 31 — Query-level comparison

Now we need to see exactly what each configuration gets right/wrong.

In [43]:
def build_query_level_comparison(
    experiment_results
):

    rows = []

    labels = list(
        experiment_results.keys()
    )

    for i in range(
        len(experiment_results[labels[0]])
    ):

        base_item = (
            experiment_results[labels[0]][i]
        )

        row = {
            "query_id":
                base_item["query_id"],
            "query":
                base_item["query"],
            "expected_document":
                base_item["expected_document"]
        }

        for label in labels:

            item = (
                experiment_results[label][i]
            )

            documents = [
                r["document"]
                for r in item["results"]
            ]

            rank = 999

            if item["expected_document"] in documents:

                rank = (
                    documents.index(
                        item["expected_document"]
                    ) + 1
                )

            row[f"{label}_rank"] = rank

        rows.append(row)

    return pd.DataFrame(rows)

## Cell 32 — Short query comparison

In [44]:
short_query_comparison = (
    build_query_level_comparison(
        short_results
    )
)

short_query_comparison.head()

,query_id,query,expected_document,A_semantic_rank,B_hybrid_rank,C_semantic_reranker_rank,D_hybrid_reranker_rank
0,1,How much vacation time do employees get?,Vacation and Sick Leave.md,1,1,1,1
1,2,Can I take leave after having a baby?,New Parent Leave.md,1,1,1,1
2,3,What holidays does the company observe?,Holiday List.md,1,1,1,1
3,4,How many days of PTO do I earn per month?,Vacation and Sick Leave.md,1,1,1,1
4,5,"I'm not feeling well today, how does sick leav...",Vacation and Sick Leave.md,1,2,1,1


## Cell 33 — Long query comparison

In [45]:
long_query_comparison = (
    build_query_level_comparison(
        long_results
    )
)

long_query_comparison.head()

,query_id,query,expected_document,A_semantic_rank,B_hybrid_rank,C_semantic_reranker_rank,D_hybrid_reranker_rank
0,1,I've been working at Clef for about six months...,Vacation and Sick Leave.md,1,1,1,1
1,2,My spouse and I are expecting a baby in a few ...,New Parent Leave.md,2,1,2,2
2,3,I recently adopted a child and I'm trying to f...,New Parent Leave.md,1,2,1,1
3,4,I have a chronic illness that sometimes makes ...,Vacation and Sick Leave.md,999,2,2,3
4,5,If I take the full twelve weeks of new parent ...,New Parent Leave.md,1,1,1,1


## Cell 34 — Find where Hybrid helps

In [46]:
hybrid_improvements_short = (
    short_query_comparison[
        (
            short_query_comparison["B_hybrid_rank"]
            <
            short_query_comparison["A_semantic_rank"]
        )
    ]
)

hybrid_improvements_long = (
    long_query_comparison[
        (
            long_query_comparison["B_hybrid_rank"]
            <
            long_query_comparison["A_semantic_rank"]
        )
    ]
)

print(
    "Short — Hybrid improved rank:",
    len(hybrid_improvements_short)
)

print(
    "Long — Hybrid improved rank:",
    len(hybrid_improvements_long)
)

Short — Hybrid improved rank: 13
Long — Hybrid improved rank: 4


## Cell 35 — Find where Hybrid hurts

In [47]:
hybrid_regressions_short = (
    short_query_comparison[
        (
            short_query_comparison["B_hybrid_rank"]
            >
            short_query_comparison["A_semantic_rank"]
        )
    ]
)

hybrid_regressions_long = (
    long_query_comparison[
        (
            long_query_comparison["B_hybrid_rank"]
            >
            long_query_comparison["A_semantic_rank"]
        )
    ]
)

print(
    "Short — Hybrid worsened rank:",
    len(hybrid_regressions_short)
)

print(
    "Long — Hybrid worsened rank:",
    len(hybrid_regressions_long)
)

Short — Hybrid worsened rank: 20
Long — Hybrid worsened rank: 4


## Cell 36 — Find where reranking helps

In [48]:
reranker_improvements_short = (
    short_query_comparison[
        (
            short_query_comparison[
                "D_hybrid_reranker_rank"
            ]
            <
            short_query_comparison[
                "B_hybrid_rank"
            ]
        )
    ]
)


reranker_improvements_long = (
    long_query_comparison[
        (
            long_query_comparison[
                "D_hybrid_reranker_rank"
            ]
            <
            long_query_comparison[
                "B_hybrid_rank"
            ]
        )
    ]
)


print(
    "Short — Reranker improved rank:",
    len(reranker_improvements_short)
)


print(
    "Long — Reranker improved rank:",
    len(reranker_improvements_long)
)

Short — Reranker improved rank: 19
Long — Reranker improved rank: 2


## Cell 37 — Find where reranking hurts

In [49]:
reranker_regressions_short = (
    short_query_comparison[
        (
            short_query_comparison[
                "D_hybrid_reranker_rank"
            ]
            >
            short_query_comparison[
                "B_hybrid_rank"
            ]
        )
    ]
)


reranker_regressions_long = (
    long_query_comparison[
        (
            long_query_comparison[
                "D_hybrid_reranker_rank"
            ]
            >
            long_query_comparison[
                "B_hybrid_rank"
            ]
        )
    ]
)


print(
    "Short — Reranker worsened rank:",
    len(reranker_regressions_short)
)

print(
    "Long — Reranker worsened rank:",
    len(reranker_regressions_long)
)

Short — Reranker worsened rank: 7
Long — Reranker worsened rank: 3


## Cell 38 — Show reranker regressions

In [50]:
print("SHORT — RERANKER REGRESSIONS")


display(
    reranker_regressions_short[
        [
            "query_id",
            "query",
            "expected_document",
            "B_hybrid_rank",
            "D_hybrid_reranker_rank"
        ]
    ]
)


print("\nLONG — RERANKER REGRESSIONS")


display(
    reranker_regressions_long[
        [
            "query_id",
            "query",
            "expected_document",
            "B_hybrid_rank",
            "D_hybrid_reranker_rank"
        ]
    ]
)

SHORT — RERANKER REGRESSIONS


,query_id,query,expected_document,B_hybrid_rank,D_hybrid_reranker_rank
5,6,Is Thanksgiving a day off at Clef?,Holiday List.md,1,999
83,84,How does Clef celebrate a new employee's first...,Welcome to Clef.md,1,999
86,87,Who is the CEO of Clef?,Handbook Introduction.md,1,2
99,100,What is Clef's core product value?,Product Manifesto.md,3,999
102,103,What are the budget categories at Clef?,Budgeting.md,3,999
107,108,What are the meeting time requirements at Clef?,Effective Meetings.md,3,999
110,111,What tools does Clef use for policy discussions?,Policy Changes.md,1,3



LONG — RERANKER REGRESSIONS


,query_id,query,expected_document,B_hybrid_rank,D_hybrid_reranker_rank
1,2,My spouse and I are expecting a baby in a few ...,New Parent Leave.md,1,2
3,4,I have a chronic illness that sometimes makes ...,Vacation and Sick Leave.md,2,3
20,21,When I'm working from home I know I should be ...,Working Remotely.md,3,999


## Cell 39 — Find persistent failures

These are queries where even the full system cannot retrieve the expected document in Top-5.

In [51]:
persistent_short = (
    short_query_comparison[
        short_query_comparison[
            "D_hybrid_reranker_rank"
        ] > 5
    ]
)


persistent_long = (
    long_query_comparison[
        long_query_comparison[
            "D_hybrid_reranker_rank"
        ] > 5
    ]
)


print(
    "Short persistent Top-5 failures:",
    len(persistent_short)
)


print(
    "Long persistent Top-5 failures:",
    len(persistent_long)
)

Short persistent Top-5 failures: 12
Long persistent Top-5 failures: 1


## Cell 40 — Save all raw ablation results

In [52]:
def save_json(data, path):


    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:


        json.dump(
            data,
            f,
            indent=2,
            ensure_ascii=False
        )




for label, results in short_results.items():


    save_json(
        results,
        OUTPUT_DIR
        / f"short_116_{label}.json"
    )




for label, results in long_results.items():


    save_json(
        results,
        OUTPUT_DIR
        / f"long_41_{label}.json"
    )


print("Raw ablation results saved.")


Raw ablation results saved.


## Cell 41 — Save comparison tables

In [53]:
comparison_df.to_csv(
    OUTPUT_DIR
    / "phase6_ablation_metrics.csv",
    index=False
)


short_query_comparison.to_csv(
    OUTPUT_DIR
    / "short_query_level_comparison.csv",
    index=False
)


long_query_comparison.to_csv(
    OUTPUT_DIR
    / "long_query_level_comparison.csv",
    index=False
)


print("Comparison tables saved.")

Comparison tables saved.


## Cell 42 — Final validation

This is important.

In [54]:
print("=" * 70)
print("PHASE 6 VALIDATION")
print("=" * 70)


assert len(chunks) == 96


assert len(short_queries) == 116
assert len(long_queries) == 41


assert len(short_results) == 4
assert len(long_results) == 4


for label in METHODS:


    assert len(short_results[label]) == 116
    assert len(long_results[label]) == 41


print("✓ Clean chunks: 96")
print("✓ Short queries: 116")
print("✓ Long queries: 41")
print("✓ Four configurations evaluated")
print("✓ All 116 short queries processed for every configuration")
print("✓ All 41 long queries processed for every configuration")


print("\nPhase 6 ablation completed successfully.")

PHASE 6 VALIDATION
✓ Clean chunks: 96
✓ Short queries: 116
✓ Long queries: 41
✓ Four configurations evaluated
✓ All 116 short queries processed for every configuration
✓ All 41 long queries processed for every configuration

Phase 6 ablation completed successfully.


## Cell 43 — Final output

In [55]:
print("=" * 80)
print("FINAL PHASE 6 ABLATION RESULTS")
print("=" * 80)


display(
    comparison_df.round(2)
)


print("\n" + "=" * 80)
print("SHORT QUERY COMPONENT ANALYSIS")
print("=" * 80)


display(
    full_component_analysis(
        short_metrics
    ).round(2)
)


print("\n" + "=" * 80)
print("LONG QUERY COMPONENT ANALYSIS")
print("=" * 80)


display(
    full_component_analysis(
        long_metrics
    ).round(2)
)

FINAL PHASE 6 ABLATION RESULTS


,Dataset,Method,Top-1 (%),Top-3 (%),Top-5 (%)
0,Short (116),A_semantic,78.45,87.93,89.66
1,Short (116),B_hybrid,71.55,91.38,93.10
2,Short (116),C_semantic_reranker,83.62,88.79,89.66
3,Short (116),D_hybrid_reranker,82.76,88.79,89.66
4,Long (41),A_semantic,87.80,95.12,95.12
5,Long (41),B_hybrid,87.80,100.00,100.00
6,Long (41),C_semantic_reranker,90.24,97.56,97.56
7,Long (41),D_hybrid_reranker,90.24,97.56,97.56



SHORT QUERY COMPONENT ANALYSIS


,Metric,Semantic,Hybrid,Semantic + Reranker,Hybrid + Reranker,Hybrid Δ,Reranker Δ on Semantic,Reranker Δ on Hybrid,Total Δ
0,Top-1,78.45,71.55,83.62,82.76,-6.90,5.17,11.21,4.31
1,Top-3,87.93,91.38,88.79,88.79,3.45,0.86,-2.59,0.86
2,Top-5,89.66,93.10,89.66,89.66,3.45,0.00,-3.45,0.00



LONG QUERY COMPONENT ANALYSIS


,Metric,Semantic,Hybrid,Semantic + Reranker,Hybrid + Reranker,Hybrid Δ,Reranker Δ on Semantic,Reranker Δ on Hybrid,Total Δ
0,Top-1,87.80,87.8,90.24,90.24,0.00,2.44,2.44,2.44
1,Top-3,95.12,100.0,97.56,97.56,4.88,2.44,-2.44,2.44
2,Top-5,95.12,100.0,97.56,97.56,4.88,2.44,-2.44,2.44
